In [1]:
# ============================================================
#  MONTE CARLO PORTFOLIO SIMULATION (FAT TAILS, LEDOIT-WOLF)
# ============================================================

!pip install yfinance pandas numpy scipy matplotlib scikit-learn==1.3 seaborn

import yfinance as yf
import numpy as np
import pandas as pd
from scipy.stats import t
from sklearn.covariance import LedoitWolf
import matplotlib.pyplot as plt

# -----------------------------
# USER SETTINGS
# -----------------------------
tickers = ['CSCO', 'EFRA', 'COST', 'GLD', 'NEE', 'MSFT', 'PAVE']
weights = np.array([1/7] * len(tickers))     # equal weight, change if you want
start_portfolio_value = 100000               # starting amount
yearly_withdrawal = 10000                    # withdrawal amount
withdraw_start_year = 3                      # start withdrawing at year 3
num_years = 10
num_sims = 10000
df_t = 5                                     # degrees of freedom for fat-tailed t-dist

# -----------------------------
# 1. FETCH DATA
# -----------------------------
data = yf.download(tickers, period="10y")['Adj Close']
data = data.dropna()
returns = np.log(data / data.shift(1)).dropna()

# -----------------------------
# 2. ANNUALIZE MEAN + COVARIANCE
# -----------------------------
mu_daily = returns.mean().values
cov_daily = returns.cov().values

# Ledoit–Wolf shrinkage (use daily, then annualize)
lw = LedoitWolf().fit(returns.values)
cov_shrink = lw.covariance_

mu_annual = mu_daily * 252
cov_annual = cov_shrink * 252

dim = len(tickers)

# -----------------------------
# 3. CHOLESKY FOR CORRELATION
# -----------------------------
chol = np.linalg.cholesky(cov_annual)

# -----------------------------
# 4. SIMULATION ENGINE
# -----------------------------
def simulate_portfolio():
    pv = start_portfolio_value
    w = weights.copy()

    for year in range(1, num_years + 1):

        # multivariate t simulation
        z = np.random.standard_normal(dim)
        chi = np.random.chisquare(df_t)
        t_sample = z / np.sqrt(chi / df_t)
        annual_return_vec = mu_annual + chol @ t_sample  # log-return vector

        # apply annual returns (geometric)
        growth_factor = np.exp(annual_return_vec)
        pv = pv * np.dot(w, growth_factor)

        # annual rebalancing BEFORE withdrawal
        w = weights.copy()

        # withdrawal at end of year (starting withdraw_start_year)
        if year >= withdraw_start_year:
            pv -= yearly_withdrawal
            if pv < 0:
                pv = 0
                break

    return pv

# -----------------------------
# 5. RUN MONTE CARLO
# -----------------------------
results = np.array([simulate_portfolio() for _ in range(num_sims)])

# -----------------------------
# 6. OUTPUT RESULTS
# -----------------------------
print("--------- RESULTS ---------")
print(f"Mean Final Value: ${results.mean():,.2f}")
print(f"Median Final Value: ${np.median(results):,.2f}")
print(f"5th Percentile: ${np.percentile(results, 5):,.2f}")
print(f"95th Percentile: ${np.percentile(results, 95):,.2f}")
print(f"Probability of Ruin (ending < 0): {np.mean(results <= 0) * 100:.2f}%")
print(f"Probability > $1,000,000: {np.mean(results >= 1_000_000) * 100:.2f}%")

# -----------------------------
# 7. HISTOGRAM
# -----------------------------
plt.figure(figsize=(10,6))
plt.hist(results, bins=80)
plt.title("Distribution of Final Portfolio Values")
plt.xlabel("Final Value")
plt.ylabel("Frequency")
plt.show()

# Save results to CSV
pd.DataFrame({"final_value": results}).to_csv("simulation_results.csv", index=False)
print("Saved results to simulation_results.csv")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 44.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user


KeyboardInterrupt: 